In [1]:
print('lkfgnsdg')

lkfgnsdg


In [5]:
%pwd
import os
os.chdir("../")
#Just went one step bcakword with this directory....

%pwd

'd:\\mlops\\Crypto_Guardian'

In [ ]:
from dataclasses import dataclass
from pathlib import Path

@dataclass
class DataCleaningConfig:
    cleaning_root: Path
    data_path: Path
    saved_data: Path
    

In [32]:
from src.Crypto.constants import *
from src.Crypto.utils.helper import read_yaml,create_directories
from src.Crypto import logger

class ConfigurationManager:
    def __init__(self,config_file_path=CONFIG_FILE_PATH,
                      params_file_path = PARAMS_FILE_PATH,
                      schema_file_path = SCHEMA_FILE_PATH):
        self.config =read_yaml( config_file_path)
        self.parmas = read_yaml(params_file_path)
        self.schema = read_yaml(schema_file_path)
        
        # create_directories([self.config.artifacts_root])
        logger.debug(f"Till Now all the Yaml Files are Read Sucessfully...✅")
    def get_data_cleaning(self)->DataCleaningConfig:
        config = self.config.data_cleaning
        create_directories([config.cleaning_root])
        
        data_cleaning_config = DataCleaningConfig(
            cleaning_root=config.cleaning_root,
            data_path=config.data_path,
            saved_data= config.saved_data
        )
        logger.debug("get_data_cleaning is working compeletely fine...✅")
        return data_cleaning_config


In [33]:
from src.Crypto import logger
import pandas as pd
from transformers import pipeline
import re
import emoji
import nltk
from nltk.corpus import stopwords


class DataCleaningComponent:
    def __init__(self,config: DataCleaningConfig):
        self.config = config
        
        self.df = pd.read_csv(self.config.data_path)
        logger.info(f"file REad Sucessfully...✅")
        print(self.df.head())
        
        # try:
        #     self.stopwords.words('english')
        #     self.STOPWORDS = set(self.stopwords.words('english'))

        #     logger.debug(f"Using the StopWords ")
        # except LookupError:
        #     nltk.download('stopwords')        
        #     self.STOPWORDS = set(self.stopwords.words('english'))
        #     logger.debug(f"Using the NLTK Download")
        
        try:
            self.STOPWORDS = set(stopwords.words('english'))
            logger.debug("Using existing StopWords")
        except LookupError:
            nltk.download('stopwords')        
            self.STOPWORDS = set(stopwords.words('english'))
            logger.debug("NLTK Stopwords downloaded")
            
    def remove_urls(self,text):
        """Removes URLs from the text."""
        self.url_pattern = re.compile(r'https?://\S+|www\.\S+')
        return self.url_pattern.sub(r'', text)

    def remove_emojis(self,text):
        """Removes emojis from the text."""
        return emoji.demojize(text, delimiters=("", "")) # Converts emojis to text like :smiley_face: then remove

    def remove_special_characters(self,text):
        """Removes special characters, keeping only alphanumeric and spaces."""
        # Remove HTML entities like &#39; first
        self.text = re.sub(r'&#\d+;', '', text)
        self.text = re.sub(r'&amp;', '', text)
        self.text = re.sub(r'&quot;', '', text)
        # Keep alphanumeric characters and spaces
        self.pattern = re.compile(f'[^a-zA-Z0-9\s]')
        return self.pattern.sub('', text)

    def remove_stopwords(self,text):
        """Removes stopwords from the text."""
        words = text.split()
        filtered_words = [word for word in words if word.lower() not in self.STOPWORDS]
        return ' '.join(filtered_words)

    def clean_text(self,text):
        """Enhanced cleaning for YouTube comments."""
        # 1. Lowercase
        text = text.lower()
        logger.info("Texts Got Lowered...")
        # Remove URLs
        text = self.remove_urls(text)
        logger.info("URLs Removed...")

        # Handle Emojis
        text = self.remove_emojis(text)
        logger.info("Emojis Are Handeled...")
        # Remove HTML entities and Special Characters
        text = self.remove_special_characters(text)
        logger.info("Special Charachters Removed Sucessfully....")
        # Remove Repeated Characters (e.g., 'looooove' -> 'love')
        text = re.sub(r'(.)\1{2,}', r'\1', text)
        logger.info("Repeaded Chrachters Are Removed Sucessfully...")
        # Remove Extra Whitespaces
        text = re.sub(r'\s+', ' ', text).strip()
        logger.info("Extra White Spaces are REmoved....")
        # Remove Stopwords
        text = self.remove_stopwords(text)
        logger.info("StopWords Removed Sucessfully...")

        return text

    def cleanning(self):
        self.df['text']= self.df['text'].astype(str).apply(self.clean_text)  
        logger.debug("All the Texts Are Cleaned... SucessFully...✅")

    def save_csv(self):
        self.df.to_csv(self.config.saved_data)
        logger.info(f"Data Saved Sucessfully... to {self.config.saved_data}")

In [34]:
#Creatign the Pipeline

try:
    cfm = ConfigurationManager()
    data_Cleaning = cfm.get_data_cleaning()
    data_Cleaning_component = DataCleaningComponent(data_Cleaning)
    data_Cleaning_component.cleanning()
    data_Cleaning_component.save_csv()
    logger.info("Pipeline Ran Sucessfully...✅")
except Exception as e:
    logger.error("Pipeline Error...❌")
    raise e

[05-05-2026 05:20:47 PM - helper - INFO - Yaml File :config\config.yaml Read Sucessfully✅]
[05-05-2026 05:20:47 PM - helper - INFO - Yaml File :params.yaml Read Sucessfully✅]
[05-05-2026 05:20:47 PM - helper - INFO - Yaml File :schema.yaml Read Sucessfully✅]
[05-05-2026 05:20:47 PM - helper - INFO - Folder Created Sucessfully: artifacts/data_cleaning]
[05-05-2026 05:20:47 PM - 674983398 - INFO - file REad Sucessfully...✅]
                   comment_id     video_id  \
0  Ugxwyq6x4LC7cSNn2N14AaABAg  3JZ_D3ELwOQ   
1  Ugzifx090RDuQCZf7Ol4AaABAg  3JZ_D3ELwOQ   
2  UgxcfPMtewKBQ4BGt9l4AaABAg  3JZ_D3ELwOQ   
3  UgzeB8b41WQO5wtLUvx4AaABAg  3JZ_D3ELwOQ   
4  UgxXmi7tm2DwumfZpMd4AaABAg  3JZ_D3ELwOQ   

                                                text     Label     Score  
0                      He was super Kangaroo Jacked!  NEGATIVE  0.990524  
1                        Arkansas don&#39;t play man  NEGATIVE  0.976599  
2                  kangaroo listens to NBA young boy  POSITIVE  0.968622